In [1]:
import pandas as pd

# Cargo el dataset original para empezar a limpiarlo
df = pd.read_json("../data/raw/streaming_users_dirty.json")

# Saco una copia para no arruinar los datos lógicos de origen
df_clean = df.copy()

print("¡Listo para empezar a limpiar!")

¡Listo para empezar a limpiar!


In [2]:
# Código 1: Relleno los minutos vacíos usando el promedio (media) de la columna
media_minutos = df_clean['monthly_watch_time_mins'].mean()
df_clean['monthly_watch_time_mins'] = df_clean['monthly_watch_time_mins'].fillna(media_minutos)

# Código 2: Relleno los géneros favoritos vacíos con la palabra "Unknown" (Desconocido)
df_clean['favorite_genre'] = df_clean['favorite_genre'].fillna('Unknown')

# Código 3: Borro las filas donde la fecha de login esté vacía (porque no se puede inventar una fecha)
df_clean = df_clean.dropna(subset=['last_login_date'])

# Código 4: Verifico que ya NO queden datos vacíos en ninguna columna
print("Datos faltantes después de limpiar:")
print(df_clean.isnull().sum())

Datos faltantes después de limpiar:
user_id                     0
age                         0
subscription_plan           0
monthly_watch_time_mins     0
country                     0
favorite_genre              0
last_login_date             0
customer_support_tickets    0
dtype: int64


In [3]:
# Evidencia detectada en el EDA: Inconsistencias en las variables categóricas.
# Homogeneizamos los textos a minúsculas, quitamos espacios y corregimos términos duplicados.

# 1. Limpieza de la columna de planes de suscripción
df_clean['subscription_plan'] = df_clean['subscription_plan'].astype(str).str.lower().str.strip()
plan_mapping = {
    'básico': 'basic', 'basico': 'basic', 'basic': 'basic',
    'estándar': 'standard', 'estandar': 'standard', 'standard': 'standard', 'std': 'standard',
    'premium': 'premium'
}
df_clean['subscription_plan'] = df_clean['subscription_plan'].map(plan_mapping)

# 2. Limpieza de la columna de géneros favoritos
df_clean['favorite_genre'] = df_clean['favorite_genre'].astype(str).str.lower().str.strip()
genre_mapping = {
    'acción': 'action', 'accion': 'action', 'action': 'action',
    'comedia': 'comedy', 'comedy': 'comedy',
    'drama': 'drama', 'drama': 'drama',
    'romance': 'romance', 'romance': 'romance',
    'thriller': 'thriller', 'thriller': 'thriller',
    'crimen': 'crime', 'crime': 'crime',
    'documental': 'documentary', 'documentary': 'documentary', 'doc': 'documentary',
    'unknown': 'unknown'
}
df_clean['favorite_genre'] = df_clean['favorite_genre'].map(genre_mapping).fillna('unknown')

print("¡Columnas categóricas unificadas con éxito!")

¡Columnas categóricas unificadas con éxito!


In [4]:
# Código 5: Guardo el dataset ya limpio en la carpeta de datos procesados
df_clean.to_json("../data/processed/streaming_users_clean.json", orient="records", indent=4)

print("¡Archivo 'streaming_users_clean.json' guardado con éxito en la carpeta processed!")

¡Archivo 'streaming_users_clean.json' guardado con éxito en la carpeta processed!


In [5]:
import pandas as pd
import os

# Creo la carpeta logs si no existe
if not os.path.exists("../logs"):
    os.makedirs("../logs")

# Datos del proceso
data_log = [
    ["1", "Estado inicial", 8160, 320, 100.0],
    ["2", "Limpieza de nulos (imputación y borrado)", 7840, 0, 96.07]
]

# Creo el DataFrame y lo guardo
df_log = pd.DataFrame(data_log, columns=["Paso", "Descripción", "Filas", "Nulos", "Retención (%)"])
df_log.to_csv("../logs/pipeline_log.csv", index=False)

print("¡Log de ETL creado en la carpeta logs!")

¡Log de ETL creado en la carpeta logs!
